# Install

In [ ]:
!pip3 install pshmodule

In [ ]:
!pip3 install pickle5

In [ ]:
!pip3 install pandas==1.5.0

In [ ]:
!pip3 install swifter

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Load

In [6]:
import sys
sys.path.append('/content/drive/MyDrive/Advertising_By_Personality/src/preprocessing')
print(sys.path)

['/content', '/env/python', '/usr/lib/python39.zip', '/usr/lib/python3.9', '/usr/lib/python3.9/lib-dynload', '', '/usr/local/lib/python3.9/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.9/dist-packages/IPython/extensions', '/root/.ipython', '/content/drive/MyDrive/Advertising_By_Personality/src/preprocessing']


In [7]:
import itertools
import swifter
import config as cfg
import pandas as pd
from tqdm import tqdm
from pshmodule.utils import filemanager as fm

In [8]:
df = fm.load(cfg.origin_data)

extension : .xlsx
Loaded 25000 records from /content/drive/MyDrive/Advertising_By_Personality/data/origin/origin.xlsx


In [9]:
df.rename(columns={'관리번호':'no', '광고 구분':'advertisement_type', '마케팅전략':'marketing_strategy', '일상정보':'daily_information', '시즌정보':'season_information', '분류':'type', '제목':'title', '본문':'content', '납품차수':'degree'}, inplace=True)

In [10]:
df.head()

,no,advertisement_type,marketing_strategy,daily_information,season_information,type,title,content,degree
0,1,NaN,NaN,NaN,NaN,원문,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰,"마지막 2일! [계절밥상 디너/주말 1만원, 런치 5천원 할인 쿠폰] 9월30일까지...",NaN
1,1,할인/쿠폰/혜택,"매력적인숫자(시간,돈)",NaN,NaN,ST,★최대 1만원★ 계절밥상 할인쿠폰,"이.틀.만! VIP 고객 디너&주말 1만원, 런치 5천원 할인 쿠폰 놓치지 마세요!",1차
2,1,할인/쿠폰/혜택,핵심정보만 전달,NaN,NaN,SF,ONLY VIP! 할인쿠폰♬,소중한 이들과 더 든든하게 먹고 싶다면?\\[계절밥상] 디너/주말 1만원+런치 5천...,1차
3,1,할인/쿠폰/혜택,호기심유발(질문),NaN,NaN,NT,이 혜택💰 놓치지 않을 거예요,"[계절밥상] 디너/주말 1O,OOO원, 런치 5,OOO원 할인\\9월 30일까지🚨",1차
4,1,할인/쿠폰/혜택,호기심유발(이모지),NaN,NaN,NF,VIP님들만을 위한 쿠폰 도착💌,계절밥상에서 든든한 한끼🍱 어때요?\\런치/디너/주말 할인 쿠폰(~9/30) 사용하...,1차


# Emoji Processing

In [11]:
import re

In [13]:
def convert_to_python_unicode(emoji_code):
    # Java, JavaScript, JSON 유니코드 이모지 코드를 찾는 정규식
    pattern = re.compile(r"(?:\\u[0-9a-fA-F]{4}){2}")
    
    def surrogate_pair(match):
        code_points = [int(x, 16) for x in match.group(0).split('\\u')[1:]]
        high, low = code_points
        code_point = 0x10000 + ((high - 0xD800) << 10) + (low - 0xDC00)
        return f"\\U{code_point:08x}"
    
    return pattern.sub(surrogate_pair, emoji_code)

In [14]:
df['title_processing'] = df.title.swifter.apply(convert_to_python_unicode)
df['content_processing'] = df.content.swifter.apply(convert_to_python_unicode)

Pandas Apply:   0%|          | 0/25000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/25000 [00:00<?, ?it/s]

In [18]:
print('\U0001f31f제일제면소 무료 쿠폰 만료 임박!')

🌟제일제면소 무료 쿠폰 만료 임박!


In [16]:
df[df.no == 4614]

,no,advertisement_type,marketing_strategy,daily_information,season_information,type,title,content,degree,title_processing,content_processing
23065,4614,NaN,NaN,NaN,NaN,원문,\ud83c\udf1f제일제면소 무료 쿠폰 만료 임박!,제일제면소\ud83c\udf5c 소불고기 우동 OR 우삼겹 비빔국수 무료 쿠폰 만료...,NaN,\U0001f31f제일제면소 무료 쿠폰 만료 임박!,제일제면소\U0001f35c 소불고기 우동 OR 우삼겹 비빔국수 무료 쿠폰 만료 임박!
23066,4614,할인/쿠폰/혜택,시간제한,NaN,NaN,ST,"째깍째깍, 다가오는 쿠폰 소멸 시간",[제일제면소] 무료 쿠폰 소/멸/임/박\\소불고기 우동 또는 우삼겹 비빔국수 공.짜...,4차,"째깍째깍, 다가오는 쿠폰 소멸 시간",[제일제면소] 무료 쿠폰 소/멸/임/박\\소불고기 우동 또는 우삼겹 비빔국수 공.짜...
23067,4614,할인/쿠폰/혜택,핵심정보만 전달,NaN,NaN,SF,고객님께만 몰래 드렸던 선물...,"제일제면소 소불고기 우동 or 우삼겹 비빔국수 무료 쿠폰!\\정성 들인 한 그릇, ...",4차,고객님께만 몰래 드렸던 선물...,"제일제면소 소불고기 우동 or 우삼겹 비빔국수 무료 쿠폰!\\정성 들인 한 그릇, ..."
23068,4614,할인/쿠폰/혜택,시간제한,NaN,NaN,NT,쿠폰도 상해요...😢,🚨만/료/임/박 [제일제면소] 소불고기 우동 또는 우삼겹 비빔국수 무료 쿠폰💌 얼른...,4차,쿠폰도 상해요...😢,🚨만/료/임/박 [제일제면소] 소불고기 우동 또는 우삼겹 비빔국수 무료 쿠폰💌 얼른...
23069,4614,할인/쿠폰/혜택,"재밌는 포맷(유행,밈 등)",NaN,NaN,NF,늦었다고 생각할 때 정말 늦었다,부지런한 우리 고객님👁️ 이번엔 늦을 리 없겠죠?(과연)(두근) 인내심 강한 제일제...,4차,늦었다고 생각할 때 정말 늦었다,부지런한 우리 고객님👁️ 이번엔 늦을 리 없겠죠?(과연)(두근) 인내심 강한 제일제...


# Reshape

In [ ]:
result = []
temp = []

for i in df.iterrows():
  temp.append([i[1]['type'], i[1]['title'] + '\n'  + i[1]['content']])
  if len(temp) == 5:
    index_combinations = list(itertools.combinations(temp, 2))
    for c in index_combinations:
      result.append([c[0][0], c[1][0], c[0][1], c[1][1]])
    for c in index_combinations:
      result.append([c[1][0], c[0][0], c[1][1], c[0][1]])
    temp = []

In [ ]:
len(result)

100000

In [ ]:
df_result = pd.DataFrame(result, columns=['ctrl1', 'ctrl2', 'input', 'label'])

In [ ]:
df_result.head(20)

,ctrl1,ctrl2,input,label
0,원문,ST,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\n마지막 2일! [계절밥상 디...,"★최대 1만원★ 계절밥상 할인쿠폰\n이.틀.만! VIP 고객 디너&주말 1만원, 런..."
1,원문,SF,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\n마지막 2일! [계절밥상 디...,ONLY VIP! 할인쿠폰♬\n소중한 이들과 더 든든하게 먹고 싶다면?\\[계절밥상...
2,원문,NT,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\n마지막 2일! [계절밥상 디...,"이 혜택💰 놓치지 않을 거예요\n[계절밥상] 디너/주말 1O,OOO원, 런치 5,O..."
3,원문,NF,[CJONE VIP] 계절밥상 1만원/5천원 할인쿠폰\n마지막 2일! [계절밥상 디...,VIP님들만을 위한 쿠폰 도착💌\n계절밥상에서 든든한 한끼🍱 어때요?\\런치/디너/...
4,ST,SF,"★최대 1만원★ 계절밥상 할인쿠폰\n이.틀.만! VIP 고객 디너&주말 1만원, 런...",ONLY VIP! 할인쿠폰♬\n소중한 이들과 더 든든하게 먹고 싶다면?\\[계절밥상...
5,ST,NT,"★최대 1만원★ 계절밥상 할인쿠폰\n이.틀.만! VIP 고객 디너&주말 1만원, 런...","이 혜택💰 놓치지 않을 거예요\n[계절밥상] 디너/주말 1O,OOO원, 런치 5,O..."
6,ST,NF,"★최대 1만원★ 계절밥상 할인쿠폰\n이.틀.만! VIP 고객 디너&주말 1만원, 런...",VIP님들만을 위한 쿠폰 도착💌\n계절밥상에서 든든한 한끼🍱 어때요?\\런치/디너/...
7,SF,NT,ONLY VIP! 할인쿠폰♬\n소중한 이들과 더 든든하게 먹고 싶다면?\\[계절밥상...,"이 혜택💰 놓치지 않을 거예요\n[계절밥상] 디너/주말 1O,OOO원, 런치 5,O..."
8,SF,NF,ONLY VIP! 할인쿠폰♬\n소중한 이들과 더 든든하게 먹고 싶다면?\\[계절밥상...,VIP님들만을 위한 쿠폰 도착💌\n계절밥상에서 든든한 한끼🍱 어때요?\\런치/디너/...
9,NT,NF,"이 혜택💰 놓치지 않을 거예요\n[계절밥상] 디너/주말 1O,OOO원, 런치 5,O...",VIP님들만을 위한 쿠폰 도착💌\n계절밥상에서 든든한 한끼🍱 어때요?\\런치/디너/...


# Save

In [ ]:
import json

In [ ]:
temp_dict = [{"ctrl1":row['ctrl1'].strip(), "ctrl2":row['ctrl2'].strip(), "input": row['input'].strip(), "label": row['label']} for _, row in df_result.iterrows()]

with open(cfg.train_data, 'w', encoding='utf-8') as f:
    for line in temp_dict:
        json_record = json.dumps(line, ensure_ascii=False)
        f.write(json_record + '\n')